# Heterogeneous Multi-Rover Earth-Moving Thesis Playground

This notebook demonstrates the **current integrated implementation** used by
`orchestrator_hybrid_multi_astar_scheduled.py`. It launches the real simulator rather
than reimplementing the algorithms in a simplified notebook model.

The playground focuses on the additions developed for the thesis:

- small and large rover types with independently configurable shovel geometry;
- rover-specific planning overlays, source-cell aggregation, path choices and reservations;
- mixed pebble sizes and material-mass planning;
- circular, rectangular, elliptical, semicircular, thin-L, thick concave-L and amorphous targets;
- arbitrary off-center target placement;
- task policies, highway/target fallback behavior and capacity-aware allocation;
- task-phase and rover-type right-of-way priorities;
- cooperative collision recovery and anomaly protection;
- structured JSONL event logs, task CSV summaries and post-run trajectory analysis.

Run the notebook from top to bottom. The simulation cell opens PyBullet and Pygame.
Close the simulation windows to finish the run, then execute the analysis section.


## 1. Quick start

1. Run **Setup** once.
2. Edit the three configuration cells: experiment, rover types, and motion/logging.
3. Run **Validate and preview**. It shows the derived cell sizes and safety footprints.
4. Run **Launch the real simulation**.
5. Close the GUI when finished and run **Analyze the latest log**.

If a GUI remains open after interrupting a cell, close it or restart the notebook kernel
before launching another simulation.


## 2. Setup


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import replace
import csv
import json
import math
import subprocess
import sys

def find_runtime(start=None):
    start = Path.cwd() if start is None else Path(start)
    for base in (start, *start.parents):
        candidates = (base, base / "Hybrid Orchestrator")
        for hybrid in candidates:
            runner = hybrid / "orchestrator_hybrid_multi_astar_scheduled.py"
            tracking = hybrid.parent / "Path Tracking"
            if runner.exists() and tracking.exists():
                return hybrid.resolve(), tracking.resolve()
    raise FileNotFoundError(
        "Could not find Hybrid Orchestrator and sibling Path Tracking folders."
    )

HYBRID, PATH_TRACKING = find_runtime()
RUNNER = HYBRID / "orchestrator_hybrid_multi_astar_scheduled.py"
PYTHON = Path(sys.executable)

for folder in (HYBRID, PATH_TRACKING):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import rover_profiles as rp
import scenario_configs as sc
from target_zones import resolve_target_zone

def run_stream(command, cwd, label="simulation"):
    command = [str(value) for value in command]
    printable = list(command)
    if "-c" in printable:
        code_index = printable.index("-c") + 1
        if code_index < len(printable):
            printable[code_index] = "<notebook configuration launcher>"
    print("Launching:", label)
    print("Working directory:", cwd)
    print("Command:", " ".join(printable))
    print("-" * 88)
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="")
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
        raise
    finally:
        print("\n" + "-" * 88)
    print("Process exited with code", return_code)
    return return_code

print("Python:", PYTHON)
print("Hybrid runtime:", HYBRID)
print("Path Tracking:", PATH_TRACKING)
print("Runner:", RUNNER)
print("Runtime ready:", RUNNER.exists())


## 3. What the planner changes for each rover

The system keeps one canonical earth-moving heatmap, then builds a profile-specific
overlay for each rover geometry. With `planning_cell_size_override=None`:

\[
\text{overlay cell size}=\frac{\text{shovel width}}{2\sqrt{2}}
\]

A wider shovel therefore changes source aggregation, push-path candidates, swept
reservations, A* clearance, collision radius and endpoint tolerance. It does **not**
automatically change task policy, capacity or right-of-way priority; those are explicit
experimental variables below.

Collision avoidance uses a conservative circular envelope around the chassis and shovel.
During interaction, a smaller numeric priority wins. Task phase dominates rover type:
`PUSH` is protected ahead of `TURN_TO_PUSH`, `APPROACH`, and `ROLLBACK`.


In [ ]:
print("Available scenarios")
print("-" * 88)
for name, scenario in sc.SCENARIOS.items():
    zone = scenario.target_zone
    print(
        f"{name:28s} shape={zone.shape:11s} center={tuple(round(v, 3) for v in zone.center)!s:18s} "
        f"area={zone.area:5.3f}  {scenario.description}"
    )

print("\nDefault rover profiles")
print("-" * 88)
print(
    f"{'type':8s} {'scale':>7s} {'shovel':>8s} {'cell':>8s} {'collision':>10s} "
    f"{'reserve':>9s} {'cap obj':>8s} {'cap mass':>9s}  tasks"
)
for name, profile in rp.ROVER_TYPES.items():
    print(
        f"{name:8s} {profile.geometry.chassis_scale:7.2f} "
        f"{profile.shovel_width:8.3f} {profile.overlay_cell_size:8.3f} "
        f"{profile.collision_radius:10.3f} {profile.reservation_radius:9.3f} "
        f"{profile.capacity_objects:8d} {profile.capacity_mass:9d}  "
        f"{sorted(profile.policy.allowed_tasks)}"
    )


## 4. Experiment configuration

`TARGET_ZONE_CENTER=None` keeps the selected scenario's original center. Set `(x, y)`
to move **any** target shape without changing its dimensions or orientation. Positive
X is right and positive Y is up in world coordinates.

Aggregate positions are random but reproducible for a fixed seed. Material mode `mass`
makes large visual pebbles contribute more planning quantity; `count` treats every
physical pebble equally.


In [ ]:
# ---------- EDIT THIS CELL ----------
EXPERIMENT_NAME = "advisor_heterogeneous_demo"

SCENARIO_NAME = "l_shape_target"
TARGET_ZONE_CENTER = (0.75, -0.60)  # None, (0.0, 0.0), or any valid (x, y)

ROVER_ASSIGNMENT = ["small", "large", "small"]  # 2, 3, or 4 entries

NUM_PEBBLES = 50
RANDOM_SEED = 73
MATERIAL_MODE = "mass"  # "mass" or "count"
PEBBLE_DISTRIBUTION = {
    "small": 0.30,
    "medium": 0.45,
    "large": 0.25,
}


## 5. Rover geometry, allocation policy and right-of-way

The default experiment keeps the same chassis for both types and only widens the large
shovel. `planning_cell_size_override=None` lets cell size follow shovel width.

Task policy examples:

- `("highway", "target")` means prefer highway tasks, then use target tasks when no
  valid highway task is available.
- `("target",)` means direct-to-target tasks only.
- To let the large rover use highways later, include `"highway"` in `allowed_tasks`,
  add it to `fallback_order`, and give it a nonzero `highway_weight`.

`right_of_way_priority` is a tie-breaker inside the same task phase. Smaller values have
higher priority; values are clamped to `[-4, 4]` by the runner.

Set target_root_sources_only=True for a rover type to restrict direct-target work to
upstream/root sources whose material cannot be collected by another source corridor.


In [ ]:
# ---------- EDIT THIS CELL ----------
ROVER_CONFIGS = {
    "small": {
        "chassis_scale": 1.00,
        "shovel_width": 0.22,
        "shovel_depth": 0.01,
        "shovel_height": 0.08,
        "shovel_offset": 0.17,
        "planning_cell_size_override": None,
        "capacity_objects": 8,
        "capacity_mass": 10,
        "allowed_tasks": ("highway", "target"),
        "fallback_order": ("highway", "target"),
        "target_weight": 1.0,
        "highway_weight": 1.0,
        "capacity_utilization_weight": 2.0,
        "minimum_capacity_utilization": 0.0,
        "target_root_sources_only": False,
        "right_of_way_priority": 0.50,
    },
    "large": {
        "chassis_scale": 1.00,
        "shovel_width": 0.33,
        "shovel_depth": 0.01,
        "shovel_height": 0.08,
        "shovel_offset": 0.17,
        "planning_cell_size_override": None,
        "capacity_objects": 12,
        "capacity_mass": 18,
        "allowed_tasks": ("target",),
        "fallback_order": ("target",),
        "target_weight": 1.4,
        "highway_weight": 0.0,
        "capacity_utilization_weight": 3.0,
        "minimum_capacity_utilization": 0.08,
        "target_root_sources_only": True,
        "right_of_way_priority": 0.00,
    },
}

# Task-phase priority always dominates the rover-type tie-breaker.
# Smaller numeric value = higher priority.
PHASE_PRIORITIES = {
    "PUSH": 0.0,
    "TURN_TO_PUSH": 10.0,
    "APPROACH": 20.0,
    "ROLLBACK": 30.0,
}


## 6. Motion, visualization, safety experiments and logging

Structured logging produces:

- JSONL: assignments, paths, phase changes, rover states, pairwise proximity,
  collision recovery, external repositioning and anomaly quarantine;
- CSV: one compact row per completed/replaced task with estimated and actual timing.

Scheduler goal-directed replans remain disabled by default because the legacy replanner
can bypass the selected earth-moving push corridor. Approach-only replanning and
cooperative collision recovery remain active.

NAVIGATION_OUTSIDE_MARGIN gives approach planning and collision recovery room beyond
the material-map edge; it does not create material cells or push tasks outside the map.


In [ ]:
# ---------- EDIT THIS CELL ----------
MAP_UPDATE_INTERVAL = 12.0
APPROACH_REPLAN_INTERVAL = 2.5
NAVIGATION_OUTSIDE_MARGIN = 1.0
PUSH_EXTRA_DISTANCE = 0.25
PHASE1_TRACKING_POINT = "shovel"  # "shovel" or "base"

V_MAX = 1.0
W_MAX = 10.0
PATH_STOP_S = 0.05
GOAL_DIST_TOL = 0.06
GOAL_RELAXED_REMAINING_S = 0.10
GOAL_RELAXED_DIST_TOL = 0.10

SHOW_3D_PATHS = False
DRAW_CONFLICT_MARKERS = False
FLOW_FIELD_VIS = "never"  # "never", "ask", "always"
EXPERIMENTAL_SCHEDULER_REPLANS = False
SCHEDULER_REPLAN_WORKERS = 0

ENABLE_EVENT_LOG = True
EVENT_LOG_POSE_INTERVAL = 0.50
LOG_DIR = HYBRID / "playground_logs" / EXPERIMENT_NAME


## 7. Validate and preview the selected experiment


In [ ]:
import matplotlib.pyplot as plt

VALID_TASKS = {rp.TARGET_TASK, rp.HIGHWAY_TASK}

def configured_profile(name, config):
    base = rp.ROVER_TYPES[name]
    geometry = replace(
        base.geometry,
        chassis_scale=float(config["chassis_scale"]),
        shovel_width=float(config["shovel_width"]),
        shovel_depth=float(config["shovel_depth"]),
        shovel_height=float(config["shovel_height"]),
        shovel_offset=float(config["shovel_offset"]),
        planning_cell_size_override=config["planning_cell_size_override"],
    )
    capabilities = replace(
        base.capabilities,
        capacity_objects=int(config["capacity_objects"]),
        capacity_mass=int(config["capacity_mass"]),
    )
    policy = replace(
        base.policy,
        allowed_tasks=frozenset(config["allowed_tasks"]),
        task_fallback_order=tuple(config["fallback_order"]),
        target_weight=float(config["target_weight"]),
        highway_weight=float(config["highway_weight"]),
        capacity_utilization_weight=float(config["capacity_utilization_weight"]),
        minimum_capacity_utilization=float(config["minimum_capacity_utilization"]),
        target_root_sources_only=bool(config["target_root_sources_only"]),
    )
    return replace(
        base,
        geometry=geometry,
        capabilities=capabilities,
        policy=policy,
        right_of_way_priority=float(config["right_of_way_priority"]),
    )

assert SCENARIO_NAME in sc.SCENARIOS, f"Unknown scenario: {SCENARIO_NAME}"
assert 2 <= len(ROVER_ASSIGNMENT) <= 4, "Use 2, 3, or 4 rovers"
assert all(name in ROVER_CONFIGS for name in ROVER_ASSIGNMENT)
assert MATERIAL_MODE in {"mass", "count"}
assert set(PEBBLE_DISTRIBUTION) == {"small", "medium", "large"}
assert all(float(value) >= 0 for value in PEBBLE_DISTRIBUTION.values())
assert sum(float(value) for value in PEBBLE_DISTRIBUTION.values()) > 0
assert set(PHASE_PRIORITIES) == {"PUSH", "TURN_TO_PUSH", "APPROACH", "ROLLBACK"}

preview_profiles = {}
for name, config in ROVER_CONFIGS.items():
    allowed = set(config["allowed_tasks"])
    fallback = tuple(config["fallback_order"])
    assert allowed and allowed <= VALID_TASKS
    assert set(fallback) <= allowed
    preview_profiles[name] = configured_profile(name, config)

selected_scenario = sc.get_scenario(SCENARIO_NAME)
if TARGET_ZONE_CENTER is not None:
    selected_scenario = selected_scenario.with_target_center(TARGET_ZONE_CENTER)

print("Experiment:", EXPERIMENT_NAME)
print("Scenario:", selected_scenario.name)
print("Target:", selected_scenario.target_zone.to_dict())
print("Rovers:", ROVER_ASSIGNMENT)
print("Material:", MATERIAL_MODE, PEBBLE_DISTRIBUTION)
print("Logs:", LOG_DIR)

print("\nDerived rover parameters")
print("-" * 112)
print(
    f"{'type':8s} {'scale':>7s} {'shovel':>8s} {'cell':>8s} {'A* radius':>10s} "
    f"{'collision':>10s} {'reserve':>9s} {'capacity':>12s} {'ROW':>7s}  task tiers"
)
for name, profile in preview_profiles.items():
    print(
        f"{name:8s} {profile.geometry.chassis_scale:7.2f} "
        f"{profile.shovel_width:8.3f} {profile.overlay_cell_size:8.3f} "
        f"{profile.astar_radius:10.3f} {profile.collision_radius:10.3f} "
        f"{profile.reservation_radius:9.3f} "
        f"{profile.capacity_objects:3d}/{profile.capacity_mass:<8d} "
        f"{profile.right_of_way_priority:7.2f}  "
        f"{profile.policy.selection_tiers()}"
    )

zone = selected_scenario.target_zone
figure, axis = plt.subplots(figsize=(7, 7))
arena = plt.Circle((0, 0), selected_scenario.env_radius, fill=False, ls="--", color="0.45")
axis.add_patch(arena)
geometry = zone.geometry
x_values, y_values = geometry.exterior.xy
axis.fill(x_values, y_values, color="tomato", alpha=0.35, label="target zone")
axis.plot(x_values, y_values, color="firebrick", lw=2)
axis.scatter([zone.center[0]], [zone.center[1]], marker="x", color="black", label="target center")
axis.set_aspect("equal")
axis.set_xlim(-selected_scenario.env_radius - 0.2, selected_scenario.env_radius + 0.2)
axis.set_ylim(-selected_scenario.env_radius - 0.2, selected_scenario.env_radius + 0.2)
axis.set_xlabel("world X [m]")
axis.set_ylabel("world Y [m]")
axis.set_title(f"{SCENARIO_NAME} at {tuple(round(v, 2) for v in zone.center)}")
axis.grid(alpha=0.25)
axis.legend(loc="upper right")
plt.show()


## 8. Launch the real integrated simulation

This cell creates a fresh Python subprocess, applies the notebook configuration to the
real immutable rover profiles and scenario registry, then calls the real runner's
`main()`. Configuration changes therefore cannot leak between experiments.

The cell remains active while the simulation runs. Close the PyBullet/Pygame windows to
finish normally. Interrupting the cell terminates the child process.


In [ ]:
launcher_lines = [
    "import sys",
    "from dataclasses import replace",
    f"hybrid = {str(HYBRID)!r}",
    f"tracking = {str(PATH_TRACKING)!r}",
    "sys.path.insert(0, hybrid)",
    "sys.path.insert(1, tracking)",
    "import rover_profiles as rp",
    "import scenario_configs as sc",
    f"configs = {ROVER_CONFIGS!r}",
    "for name, config in configs.items():",
    "    base = rp.ROVER_TYPES[name]",
    "    geometry = replace(",
    "        base.geometry,",
    "        chassis_scale=float(config['chassis_scale']),",
    "        shovel_width=float(config['shovel_width']),",
    "        shovel_depth=float(config['shovel_depth']),",
    "        shovel_height=float(config['shovel_height']),",
    "        shovel_offset=float(config['shovel_offset']),",
    "        planning_cell_size_override=config['planning_cell_size_override'],",
    "    )",
    "    capabilities = replace(",
    "        base.capabilities,",
    "        capacity_objects=int(config['capacity_objects']),",
    "        capacity_mass=int(config['capacity_mass']),",
    "    )",
    "    policy = replace(",
    "        base.policy,",
    "        allowed_tasks=frozenset(config['allowed_tasks']),",
    "        task_fallback_order=tuple(config['fallback_order']),",
    "        target_weight=float(config['target_weight']),",
    "        highway_weight=float(config['highway_weight']),",
    "        capacity_utilization_weight=float(config['capacity_utilization_weight']),",
    "        minimum_capacity_utilization=float(config['minimum_capacity_utilization']),",
    "        target_root_sources_only=bool(config['target_root_sources_only']),",
    "    )",
    "    rp.ROVER_TYPES[name] = replace(",
    "        base, geometry=geometry, capabilities=capabilities, policy=policy,",
    "        right_of_way_priority=float(config['right_of_way_priority']),",
    "    )",
    f"scenario_name = {SCENARIO_NAME!r}",
    "base_scenario = sc.SCENARIOS[scenario_name]",
    f"sc.SCENARIOS[scenario_name] = replace(base_scenario, pebble_distribution={PEBBLE_DISTRIBUTION!r})",
    "import orchestrator_hybrid_multi_astar_scheduled as runner",
    f"runner.TARGET_ZONE_CENTER = {TARGET_ZONE_CENTER!r}",
    f"runner.PROTECTED_PHASE_PRIORITIES.clear()",
    f"runner.PROTECTED_PHASE_PRIORITIES.update({PHASE_PRIORITIES!r})",
    "runner.main()",
]
launcher_code = "\n".join(launcher_lines)

command = [
    PYTHON, "-u", "-c", launcher_code,
    "--scenario", SCENARIO_NAME,
    "--rovers", len(ROVER_ASSIGNMENT),
    "--rover-profiles", ",".join(ROVER_ASSIGNMENT),
    "--pebbles", NUM_PEBBLES,
    "--seed", RANDOM_SEED,
    "--material-mode", MATERIAL_MODE,
    "--map-interval", MAP_UPDATE_INTERVAL,
    "--phase1-tracking-point", PHASE1_TRACKING_POINT,
    "--flow-field-vis", FLOW_FIELD_VIS,
    "--push-extra-distance", PUSH_EXTRA_DISTANCE,
    "--approach-replan-interval", APPROACH_REPLAN_INTERVAL,
    "--navigation-outside-margin", NAVIGATION_OUTSIDE_MARGIN,
    "--v-max", V_MAX,
    "--w-max", W_MAX,
    "--path-stop-s", PATH_STOP_S,
    "--goal-dist-tol", GOAL_DIST_TOL,
    "--goal-relaxed-remaining-s", GOAL_RELAXED_REMAINING_S,
    "--goal-relaxed-dist-tol", GOAL_RELAXED_DIST_TOL,
    "--scheduler-replan-workers", SCHEDULER_REPLAN_WORKERS,
    "--event-log-pose-interval", EVENT_LOG_POSE_INTERVAL,
]

command.append("--draw-execution-paths" if SHOW_3D_PATHS else "--no-draw-execution-paths")
command.append("--draw-conflicts" if DRAW_CONFLICT_MARKERS else "--no-draw-conflicts")
command.append(
    "--experimental-scheduler-replans"
    if EXPERIMENTAL_SCHEDULER_REPLANS
    else "--no-scheduler-replans"
)
if ENABLE_EVENT_LOG:
    command.extend(["--event-log-dir", LOG_DIR])
else:
    command.append("--no-event-log")

LAST_RETURN_CODE = run_stream(command, cwd=HYBRID, label=EXPERIMENT_NAME)


## 9. Analyze the latest structured log

Run this after the simulation process ends. The report summarizes:

- task allocation by rover and task type;
- actual duration versus the estimated interval;
- collision recovery winner/yielder events;
- safety-control state changes;
- external mouse repositioning and anomaly/quarantine events;
- final delivered material;
- rover XY trajectories over the target geometry.

A short run may have assignments but no completed task rows; the cell handles that case.


In [ ]:
latest_file = LOG_DIR / "LATEST_RUN.txt"
if not latest_file.exists():
    raise FileNotFoundError(
        f"No log pointer found at {latest_file}. Run a logged simulation first."
    )

latest_paths = {}
for line in latest_file.read_text(encoding="utf-8").splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        latest_paths[key.strip()] = Path(value.strip())

jsonl_path = latest_paths["JSONL"]
summary_path = latest_paths["TASKS_CSV"]
events = [
    json.loads(line)
    for line in jsonl_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
with summary_path.open("r", encoding="utf-8", newline="") as stream:
    task_rows = list(csv.DictReader(stream))

event_counts = Counter(event["event"] for event in events)
run_config = next((event for event in events if event["event"] == "RUN_CONFIG"), None)
assignments = [event for event in events if event["event"] == "TASK_ASSIGNED"]
recoveries = [
    event for event in events
    if event["event"] in {"COLLISION_RECOVERY_CHANGED", "COLLISION_RECOVERY_ENDED"}
]
control_changes = [
    event for event in events if event["event"] == "CONTROL_INTERACTION_CHANGED"
]
anomalies = [
    event for event in events
    if event["event"] in {
        "ROVER_QUARANTINED",
        "ROVER_OUTSIDE_NOMINAL_ARENA",
        "ROVER_EXTERNAL_REPOSITION",
        "APPROACH_REPLAN_FAILED",
    }
]
material_events = [event for event in events if event["event"] == "MATERIAL_PROGRESS"]

print("JSONL:", jsonl_path)
print("Task CSV:", summary_path)
print("Events:", len(events), "Completed/replaced task rows:", len(task_rows))
print("\nMost frequent events")
for name, count in event_counts.most_common(15):
    print(f"  {name:34s} {count:6d}")

if run_config:
    print("\nRun configuration")
    print("  scenario:", run_config.get("scenario_name"))
    print("  target:", run_config.get("target_zone"))
    print("  material mode:", run_config.get("material_value_mode"))
    print("  pebble summary:", run_config.get("pebble_summary"))
    print("  rovers:")
    for rover in run_config.get("rovers", []):
        print("   ", rover)

allocation_counts = Counter(
    (event.get("agent_id"), event.get("rover_type"), event.get("task_type"))
    for event in assignments
)
print("\nTask assignments")
if allocation_counts:
    for key, count in sorted(allocation_counts.items(), key=lambda item: str(item[0])):
        print(f"  agent={key[0]} type={key[1]} task={key[2]}: {count}")
else:
    print("  No TASK_ASSIGNED events.")

print("\nTask timing")
if task_rows:
    print(
        f"{'agent':6s} {'type':8s} {'task':9s} {'actual':>9s} "
        f"{'estimate':>19s} {'inside?':>8s} {'outcome'}"
    )
    for row in task_rows:
        actual = float(row["actual_duration_s"])
        lower = float(row["estimated_lower_s"]) if row["estimated_lower_s"] else float("nan")
        upper = float(row["estimated_upper_s"]) if row["estimated_upper_s"] else float("nan")
        inside = math.isfinite(lower) and math.isfinite(upper) and lower <= actual <= upper
        estimate = f"[{lower:.1f}, {upper:.1f}]" if math.isfinite(lower) else "n/a"
        print(
            f"{row['agent_id']:6s} {row['rover_type']:8s} {row['task_type']:9s} "
            f"{actual:9.2f} {estimate:>19s} {str(inside):>8s} {row['outcome']}"
        )
else:
    print("  No finished task rows. Run longer or close after a task completes.")

yielder_counts = Counter()
winner_counts = Counter()
for event in recoveries:
    if event.get("winner_id"):
        winner_counts[event["winner_id"]] += 1
    if event.get("yielder_id"):
        yielder_counts[event["yielder_id"]] += 1

print("\nCollision/safety interactions")
print("  recovery events:", len(recoveries))
print("  control-state changes:", len(control_changes))
print("  recovery winners:", dict(winner_counts))
print("  recovery yielders:", dict(yielder_counts))

print("\nAnomalies and manual intervention")
if anomalies:
    for event in anomalies:
        print(
            f"  t={event.get('sim_time', 0):7.2f} "
            f"{event.get('event'):30s} agent={event.get('agent_id')} "
            f"details={{{', '.join(f'{k}={v}' for k, v in event.items() if k not in {'run_id', 'wall_time', 'sim_time', 'event'})}}}"
        )
else:
    print("  None logged.")

if material_events:
    final_material = material_events[-1]
    print("\nFinal material progress")
    for key in (
        "delivered_count",
        "delivered_material_mass",
        "remaining_count",
        "remaining_material_mass",
    ):
        print(f"  {key}: {final_material.get(key)}")

rover_states = defaultdict(list)
for event in events:
    if event["event"] == "ROVER_STATE" and event.get("pose"):
        rover_states[event["agent_id"]].append(event["pose"])

if rover_states and run_config:
    figure, axis = plt.subplots(figsize=(8, 8))
    environment_radius = float(run_config.get("env_radius", 3.0))
    axis.add_patch(
        plt.Circle((0, 0), environment_radius, fill=False, ls="--", color="0.5")
    )
    target = resolve_target_zone(run_config["target_zone"])
    target_x, target_y = target.geometry.exterior.xy
    axis.fill(target_x, target_y, color="tomato", alpha=0.30, label="target")
    for agent_id, poses in sorted(rover_states.items()):
        axis.plot(
            [pose[0] for pose in poses],
            [pose[1] for pose in poses],
            lw=1.8,
            label=agent_id,
        )
        axis.scatter([poses[0][0]], [poses[0][1]], marker="o", s=35)
        axis.scatter([poses[-1][0]], [poses[-1][1]], marker="x", s=45)
    axis.set_aspect("equal")
    axis.set_xlim(-environment_radius - 0.25, environment_radius + 0.25)
    axis.set_ylim(-environment_radius - 0.25, environment_radius + 0.25)
    axis.set_xlabel("world X [m]")
    axis.set_ylabel("world Y [m]")
    axis.set_title("Logged rover trajectories (circle=start, x=end)")
    axis.grid(alpha=0.25)
    axis.legend()
    plt.show()


## 10. Suggested advisor experiments

**A. Rover-specific overlays and capacity**

Keep `ROVER_ASSIGNMENT = ["small", "large", "small"]`, use `MATERIAL_MODE="mass"`,
and compare the printed `[OVERLAY]` lines. The large rover should show a larger cell size,
wider source aggregation and a larger capacity.

**B. Target-shape comparison**

Keep seed, rover assignment and pebble distribution fixed. Run
`rectangle_target`, `ellipse_target`, `semicircle_target`, `l_shape_target`, and
`amorphous_target`. Use the same `TARGET_ZONE_CENTER` to isolate shape effects.

**C. Off-center collection**

Select any shape and set `TARGET_ZONE_CENTER=(1.0, -0.7)`. The same exact target geometry
is translated, while randomly deployed aggregates remain distributed over the arena.

**D. Task specialization**

Use the defaults: small rovers prefer highways and fall back to direct target tasks;
the large rover performs direct target tasks only. Then enable highway tasks for the
large profile to compare division of labor.

**E. Right-of-way and protected pushing**

Give one rover type a smaller `right_of_way_priority`. Observe that this decides ties
within a phase, while a rover already pushing retains priority over an approaching rover.
Review `COLLISION_RECOVERY_CHANGED` events afterward.

**F. Geometry sensitivity**

Change only the large shovel width. The validation table shows automatic changes to cell
size, A* radius, collision radius and reservation radius. `planning_cell_size_override`
can intentionally decouple planning resolution from shovel width.

**G. Root-source task policy**

Toggle target_root_sources_only for either rover type. When enabled, direct-target
tasks start only at source cells with no upstream collection corridor.

**H. Failure diagnostics**

If a rover is moved manually with the mouse, the logger records
`ROVER_EXTERNAL_REPOSITION` and reprojects path progress. Check the anomaly section for
out-of-arena, failed-replan or quarantine events.
